# 04 · Gold aggregations (REQ-03)
**Purpose:** Summarise the Silver table into 3 small Delta tables, one per business question.
Gold is rebuilt from Silver on every run (overwrite), because the tables are small and always reflect all clean data.

In [0]:
%run ./00_config

# Scratch / Practice Notebook

This notebook is my personal scratch space for experimenting and practicing new PySpark/Spark SQL concepts while building this project.

**It is NOT part of the graded pipeline.** The real pipeline is: `00_config` -> `01_setup_landing_zone` -> `02_bronze_ingestion` -> `03_silver_cleaning` -> `04_gold_aggregations` -> `05_analytics` -> `06_incremental_test`.

# 00 · Config
All project names and paths in one place. Every other notebook loads this with `%run ./00_config`.

✅ Config loaded. Landing folder: /Volumes/workspace/nyc_taxi/pipeline_files/landing


In [0]:
from pyspark.sql import functions as F

# Open the clean Silver table (normal read, not streaming)
silver = spark.table(SILVER_TABLE)

print("Silver rows:", f"{silver.count():,}")

Silver rows: 50,169,175


In [0]:
demand_counts = (
    silver
        .groupBy("day_type", "pickup_hour")
        .agg(
            F.count("*").alias("total_trips"),
            F.countDistinct("pickup_date").alias("num_days"),
        )
        .orderBy("day_type", "pickup_hour")
)

display(demand_counts)

day_type,pickup_hour,total_trips,num_days
Weekday,0,793113,152
Weekday,1,462301,152
Weekday,2,279110,152
Weekday,3,186638,152
Weekday,4,194068,152
Weekday,5,362331,152
Weekday,6,934881,152
Weekday,7,1637277,152
Weekday,8,1993923,152
Weekday,9,1877022,152


In [0]:
demand_per_day = demand_counts.withColumn(
    "avg_trips_per_day",
    F.round(F.col("total_trips") / F.col("num_days"), 0)
)

display(demand_per_day)

day_type,pickup_hour,total_trips,num_days,avg_trips_per_day
Weekday,0,793113,152,5218.0
Weekday,1,462301,152,3041.0
Weekday,2,279110,152,1836.0
Weekday,3,186638,152,1228.0
Weekday,4,194068,152,1277.0
Weekday,5,362331,152,2384.0
Weekday,6,934881,152,6151.0
Weekday,7,1637277,152,10772.0
Weekday,8,1993923,152,13118.0
Weekday,9,1877022,152,12349.0


In [0]:
# Save as a named Delta table (replace it every time we rebuild)
(
    demand_per_day
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_DEMAND_TABLE)
)

print("✅ Saved:", GOLD_DEMAND_TABLE)

✅ Saved: workspace.nyc_taxi.gold_demand_by_hour


In [0]:
display(spark.sql(f"SELECT * FROM {GOLD_DEMAND_TABLE} ORDER BY day_type, pickup_hour"))

day_type,pickup_hour,total_trips,num_days,avg_trips_per_day
Weekday,0,793113,152,5218.0
Weekday,1,462301,152,3041.0
Weekday,2,279110,152,1836.0
Weekday,3,186638,152,1228.0
Weekday,4,194068,152,1277.0
Weekday,5,362331,152,2384.0
Weekday,6,934881,152,6151.0
Weekday,7,1637277,152,10772.0
Weekday,8,1993923,152,13118.0
Weekday,9,1877022,152,12349.0


In [0]:
total_rides = silver.count()   # all clean rides, used for the "share" column

payment_summary = (
    silver
        .groupBy("payment_method")
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
            F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
            F.round(F.avg("tip_percent"), 2).alias("avg_tip_percent"),
            F.round(F.avg(F.when(F.col("tip_amount") > 0, 1).otherwise(0)) * 100, 1)
                .alias("pct_trips_with_tip"),
        )
        .withColumn("share_of_trips_pct",
                    F.round(F.col("total_trips") / total_rides * 100, 2))
        .orderBy(F.desc("total_trips"))
)

display(payment_summary)

payment_method,total_trips,avg_fare,avg_tip,avg_tip_percent,pct_trips_with_tip,share_of_trips_pct
Credit card,36335438,13.24,2.98,25.44,96.0,72.43
Cash,13557412,12.32,0.0,0.0,0.0,27.02
No charge,184425,21.49,0.0,0.01,0.1,0.37
Dispute,58354,14.18,0.0,0.03,0.1,0.12
Unknown,33546,36.5,0.06,0.1,0.5,0.07


In [0]:
(
    payment_summary
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_PAYMENT_TABLE)
)

print("✅ Saved:", GOLD_PAYMENT_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_PAYMENT_TABLE}"))

✅ Saved: workspace.nyc_taxi.gold_payment_tips


payment_method,total_trips,avg_fare,avg_tip,avg_tip_percent,pct_trips_with_tip,share_of_trips_pct
Credit card,36335438,13.24,2.98,25.44,96.0,72.43
Cash,13557412,12.32,0.0,0.0,0.0,27.02
No charge,184425,21.49,0.0,0.01,0.1,0.37
Dispute,58354,14.18,0.0,0.03,0.1,0.12
Unknown,33546,36.5,0.06,0.1,0.5,0.07


In [0]:
silver = spark.table(SILVER_TABLE)

vendor_summary = (
    silver
        .groupBy("vendor_name")
        .agg(
            F.count("*").alias("total_trips")
        )
)

display(vendor_summary)

vendor_name,total_trips
VeriFone,31222947
Unknown vendor,298660
Creative Mobile Technologies,18647568


In [0]:
vendor_summary = (
    silver
        .groupBy("vendor_name")
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
            F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed_mph"),
            F.round(F.avg("tip_percent"), 2).alias("avg_tip_percent")
        )
)

display(vendor_summary)

vendor_name,total_trips,avg_fare,avg_distance_miles,avg_speed_mph,avg_tip_percent
VeriFone,31222947,13.21,3.09,12.03,18.09
Unknown vendor,298660,15.02,3.62,11.76,15.95
Creative Mobile Technologies,18647568,12.72,2.85,14.66,19.02


In [0]:
total_rides = silver.count()

vendor_summary = vendor_summary.withColumn(
    "share_of_trips_pct",
    F.round(F.col("total_trips") / F.lit(total_rides) * 100, 2)
)

display(vendor_summary)


vendor_name,total_trips,avg_fare,avg_distance_miles,avg_speed_mph,avg_tip_percent,share_of_trips_pct
VeriFone,31222947,13.21,3.09,12.03,18.09,62.24
Unknown vendor,298660,15.02,3.62,11.76,15.95,0.6
Creative Mobile Technologies,18647568,12.72,2.85,14.66,19.02,37.17


In [0]:
(
    vendor_summary
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_VENDOR_TABLE)
)
print("✅ Saved:", GOLD_VENDOR_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_VENDOR_TABLE}"))

✅ Saved: workspace.nyc_taxi.gold_vendor_comparison


vendor_name,total_trips,avg_fare,avg_distance_miles,avg_speed_mph,avg_tip_percent,share_of_trips_pct
VeriFone,31222947,13.21,3.09,12.03,18.09,62.24
Unknown vendor,298660,15.02,3.62,11.76,15.95,0.6
Creative Mobile Technologies,18647568,12.72,2.85,14.66,19.02,37.17
